In [11]:
import sys
import numpy as np
import torch 
# still do preprocessing in scipy
import scipy.sparse as sp
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from importlib import reload
# get UMAP
import umap

In [20]:
# Specify the directory containing your script
leaflet_repo = '/gpfs/commons/home/kisaev/Leaflet-private/src/clustering/'

# Append this directory to sys.path
sys.path.append(leaflet_repo)

In [22]:
import Leaflet_load_cluster_data_03 as llc 

In [15]:
torch.manual_seed(42)

# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

float_type = { 
    "device" : device, 
    "dtype" : torch.float, # save memory
}

cpu


### Read in human muscle data 

In [19]:
leaflet_repo

'/gpfs/commons/home/kisaev/Leaflet-private/src/clustering/Leaflet_load_cluster_data_03.py'

In [23]:
input_files_folder = '/gpfs/commons/projects/CZI-tabula-sapiens/Leaflet-Analysis/Leaflet-Intron-Clusters/Muscle_Yes/'

# convert data to Leaflet required input formats 
final_data, coo_counts_sparse, coo_cluster_sparse, cell_ids_conversion, junction_ids_conversion = llc.load_cluster_data(
    input_folder = input_files_folder, max_intron_count=5000, has_genes="yes") 

# add cluster to final_data 
final_data = final_data.merge(junction_ids_conversion, on=["junction_id_index"], how="left")
cell_index_tensor, junc_index_tensor, my_data = llc.make_torch_data(final_data, **float_type)
simple_data_human = final_data[["cell_id_index", "Cluster", "cell_type", "junction_id_index", "juncratio", "junc_count", "cluster_count",  "junction_id", "gene_id"]]

Reading in data from folder ...
/gpfs/commons/projects/CZI-tabula-sapiens/Leaflet-Analysis/Leaflet-Intron-Clusters/Muscle_Yes/
Finished reading in data from folder ...
['mesenchymal stem cell' 'macrophage' 't cell'
 'cd8-positive, alpha-beta t cell' 'tendon cell'
 'endothelial cell of lymphatic vessel' 'capillary endothelial cell'
 'endothelial cell of artery' 'fast muscle cell'
 'skeletal muscle satellite stem cell' 'slow muscle cell'
 'endothelial cell of vascular tree' 'mature nk t cell' 'pericyte cell'
 'mesothelial cell' 'cd4-positive, alpha-beta t cell' 'smooth muscle cell'
 'erythrocyte' 'mast cell']
5009
239882
The maximum junction count was initially:  825686
1359
The maximum junction count is now:  5000
                                       cell_id  Cluster  Cluster_Counts  \
0  B107909_A11_S239.homo.gencode.v30.ERCC.chrM      146             191   
1  B107909_A11_S239.homo.gencode.v30.ERCC.chrM      146             191   
2  B107909_A11_S239.homo.gencode.v30.ERCC.chrM      

/gpfs/commons/home/kisaev/Leaflet-private/src/clustering/Leaflet_load_cluster_data_03.py:45: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  ycount_lookup = torch.sparse_coo_tensor(


### Read in mouse muscle data

In [24]:
input_files_folder = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain/model_input/"

# convert data to Leaflet required input formats 
final_data, coo_counts_sparse, coo_cluster_sparse, cell_ids_conversion, junction_ids_conversion = llc.load_cluster_data(
    input_folder = input_files_folder, max_intron_count=5000, has_genes="yes") 

# add cluster to final_data 
final_data = final_data.merge(junction_ids_conversion, on=["junction_id_index"], how="left")
cell_index_tensor, junc_index_tensor, my_data = llc.make_torch_data(final_data, **float_type)
simple_data_mouse = final_data[["cell_id_index", "Cluster", "cell_type", "junction_id_index", "juncratio", "junc_count", "cluster_count",  "junction_id", "gene_id"]]

Reading in data from folder ...
/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain/model_input/
Finished reading in data from folder ...
['Limb_Muscle_skeletal_muscle_satellite_cell' 'Limb_Muscle_macrophage'
 'Limb_Muscle_endothelial_cell' 'Limb_Muscle_mesenchymal_stem_cell'
 'Limb_Muscle_B_cell' 'Limb_Muscle_T_cell']
1090
158383
The maximum junction count was initially:  441914
326
The maximum junction count is now:  4964
                                             cell_id  Cluster  Cluster_Counts  \
0  A1-B002765-3_38_F-1-1_Limb_Muscle_skeletal_mus...      141               5   
1  A1-B002765-3_38_F-1-1_Limb_Muscle_skeletal_mus...      141               5   
2  A1-B002765-3_38_F-1-1_Limb_Muscle_skeletal_mus...      142              13   
3  A1-B002765-3_38_F-1-1_Limb_Muscle_skeletal_mus...      143              23   
4  A1-B002765-3_38_F-1-1_Limb_Muscle_skeletal_mus...      144              14   

             junction_id  gene_id  junc_count  \
0  1_1712540

### Let's get some basic summaries of events observed in each specie muscle tissue 

In [25]:
# calculate the number of singletons in the data (number of junctions in cluster == 1)
clusts_juncs = simple_data_human[["Cluster", "junction_id"]].drop_duplicates()
clusts_juncs = clusts_juncs.groupby("Cluster").size().reset_index(name='counts')
singletons = clusts_juncs[clusts_juncs['counts'] == 1]
not_singletons = clusts_juncs[clusts_juncs['counts'] != 1]
print("Number of singletons in human data: ", singletons.shape[0])
print("Number of non-singletons in human data: ", not_singletons.shape[0])

Number of singletons in human data:  87808
Number of non-singletons in human data:  27248


In [26]:
# calculate the number of singletons in the data (number of junctions in cluster == 1)
clusts_juncs = simple_data_mouse[["Cluster", "junction_id"]].drop_duplicates()
clusts_juncs = clusts_juncs.groupby("Cluster").size().reset_index(name='counts')
singletons = clusts_juncs[clusts_juncs['counts'] == 1]
print("Number of singletons in mouse data: ", singletons.shape[0])
print("Number of non-singletons in mouse data: ", clusts_juncs.shape[0] - singletons.shape[0])

Number of singletons in mouse data:  105418
Number of non-singletons in mouse data:  12106


### Find orthologs genes between human and mouse using pyensembl

In [27]:
simple_data_mouse

,cell_id_index,Cluster,cell_type,junction_id_index,juncratio,junc_count,cluster_count,junction_id,gene_id
0,0,141,Limb_Muscle_skeletal_muscle_satellite_cell,0,0.2,1.0,5,1_171254063_171256147,Adamts4
1,0,141,Limb_Muscle_skeletal_muscle_satellite_cell,1,0.8,4.0,5,1_171256434_171256527,Adamts4
2,0,142,Limb_Muscle_skeletal_muscle_satellite_cell,2,1.0,13.0,13,1_171256714_171256954,Adamts4
3,0,143,Limb_Muscle_skeletal_muscle_satellite_cell,3,1.0,23.0,23,1_171257130_171257718,Adamts4
4,0,144,Limb_Muscle_skeletal_muscle_satellite_cell,4,1.0,14.0,14,1_171257894_171258751,Adamts4
...,...,...,...,...,...,...,...,...,...
15016417,1089,62476,Limb_Muscle_B_cell,156079,0.0,0.0,7,9_88715749_88731299,Gm2382
15016418,1089,46597,Limb_Muscle_B_cell,156119,0.0,0.0,69,7_141402069_141402743,Taldo1
15016419,1089,77669,Limb_Muscle_B_cell,156127,0.0,0.0,48,11_6508294_6510147,Myo1g
15016420,1089,57078,Limb_Muscle_B_cell,156138,0.0,0.0,89,8_22578100_22580339,Vdac3


In [28]:
import pyensembl
from pyensembl import EnsemblRelease

In [29]:
import requests

In [30]:
# Load Ensembl releases for human and mouse
ensembl_human = EnsemblRelease(93, species='homo_sapiens')  # Double check actual release used in the gtf file
ensembl_mouse = EnsemblRelease(93, species='mus_musculus')

In [ ]:
#gene = ensembl_mouse.gene_by_id(gene_id='ENSMUSG00000006403')
#gene = ensembl_human.gene_by_id(gene_id='ENSG00000139618')

In [32]:
# orthologues genes file
import pandas as pd 
orthos = "/gpfs/commons/home/kisaev/ensembl_TS_TM_mart_export.txt"
orthos = pd.read_csv(orthos, sep='\t')
orthos.head()

,Gene stable ID,Gene stable ID version,Transcript stable ID,Transcript stable ID version,Mouse gene name,Mouse gene stable ID,"Mouse orthology confidence [0 low, 1 high]",Mouse homology type,Last common ancestor with Mouse
0,ENSG00000198888,ENSG00000198888.2,ENST00000361390,ENST00000361390.2,mt-Nd1,ENSMUSG00000064341,1,ortholog_one2one,Euarchontoglires
1,ENSG00000198763,ENSG00000198763.3,ENST00000361453,ENST00000361453.3,mt-Nd2,ENSMUSG00000064345,1,ortholog_one2one,Euarchontoglires
2,ENSG00000198804,ENSG00000198804.2,ENST00000361624,ENST00000361624.2,mt-Co1,ENSMUSG00000064351,1,ortholog_one2one,Euarchontoglires
3,ENSG00000198712,ENSG00000198712.1,ENST00000361739,ENST00000361739.1,mt-Co2,ENSMUSG00000064354,1,ortholog_one2one,Mammalia
4,ENSG00000228253,ENSG00000228253.1,ENST00000361851,ENST00000361851.1,mt-Atp8,ENSMUSG00000064356,0,ortholog_one2one,Euarchontoglires


In [34]:
# make a dataframe of human genes in simple_data_human
human_genes = simple_data_human.gene_id.unique()
human_genes

array(['ENSG00000023902.14', 'ENSG00000065135.12', 'ENSG00000065978.19',
       ..., 'ENSG00000267927.1', 'ENSG00000126251.7', 'ENSG00000185897.7'],
      dtype=object)

In [1]:
# remove "." from human genes 
cleaned_human_genes = [gene.replace(".", "") for gene in human_genes]
cleaned_human_genes

NameError: name 'human_genes' is not defined